In [20]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# ===================== 配置 =====================
FILE_PATH = "news_诈骗类型分析结果content.xlsx"
N_TOPICS = 5  # 想分几类就改几类（3~5类最合适）
# ====================================================

# 1. 读取数据
df = pd.read_excel(FILE_PATH)
df_clean = df[df["诈骗类型分析"].notna()].copy()

# 2. 提取文本
texts = df_clean["诈骗类型分析"].astype(str).tolist()

# 3. 文本向量化
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

# 4. 训练 LDA 模型（正确用法）
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42)
lda.fit(X)  # 这里只用 fit，不是 fit_predict

# 5. 预测每个文本属于哪个主题（正确写法！）
topic_distribution = lda.transform(X)
df_clean["LDA_聚类主题"] = [f"诈骗主题_{i+1}" for i in topic_distribution.argmax(axis=1)]

# 6. 把结果写回原 Excel（新增一列，不破坏原数据）
df.loc[df_clean.index, "LDA_聚类主题"] = df_clean["LDA_聚类主题"]
df.to_excel(FILE_PATH, index=False)

# ===================== 输出结果 =====================
print("✅ LDA 聚类完成！")
print(f"✅ 已在 Excel 新增一列：LDA_聚类主题")

print("\n" + "="*50)
print("📊 LDA 聚类结果预览")
print("="*50)

for i in range(N_TOPICS):
    theme = f"诈骗主题_{i+1}"
    group = df_clean[df_clean["LDA_聚类主题"] == theme]
    types = group["诈骗类型分析"].unique()
    print(f"\n{theme}")
    print(" 包含：" + "、".join(types[:5]))

✅ LDA 聚类完成！
✅ 已在 Excel 新增一列：LDA_聚类主题

📊 LDA 聚类结果预览

诈骗主题_1
 包含：打击治理、4. 虚假贷款、虚假广告、虚假捐款、虚假贷款

诈骗主题_2
 包含：10. 其他（假冒供应商电邮诈骗）、其他（食品掺假诈骗）、其他（职务侵占）、其他（教育机构与企业合作诈骗）、其他（洗黑钱及诈骗）

诈骗主题_3
 包含：其他（历史问题采访邀请）、10. 其他（猜猜我是谁电话诈骗）、虚假投资理财、其他（AI图片欺诈）、10. 其他（充电宝病毒诈骗）

诈骗主题_4
 包含：3. 刷单返利、杀猪盘、其他（贪污腐败、利益输送）、冒充熟人/领导、刷单返利

诈骗主题_5
 包含：破案新闻、5. 冒充客服、预警防范、分析失败、冒充客服


In [24]:
import pandas as pd
import json
import time
from zhipuai import ZhipuAI

# ===================== 你的配置 =====================
API_KEY = "aec8c161eae440bf9a03fb79803594c0.7uGk8vDg8uOIhPrU"
FILE_PATH = "news_诈骗类型分析结果content.xlsx"
OUTPUT_FILE = "news_诈骗类型_受害者画像结果.xlsx"
MODEL_NAME = "glm-4-flash"
# =====================================================

client = ZhipuAI(api_key=API_KEY)

def extract_victim_profile(news_content):
    if pd.isna(news_content) or len(str(news_content)) < 50:
        return [None] * 7  # 固定7个
    
    prompt = f"""你是一位犯罪心理分析专家。请分析下面这篇诈骗新闻报道，提取受害者的画像信息。
请注意：新闻报道中可能包含警方的防范建议、报警电话等无关信息，请忽略这些内容，仅关注受害者。

新闻报道：
"{news_content}"

请以 JSON 格式输出提取结果，包含以下字段（如果文中未提及，填 null）：
{{
    "victim_basic": {{
        "age": "年龄或范围（如：20-30岁）",
        "gender": "性别",
        "occupation": "职业（如：大学生、家庭主妇、企业财务）"
    }},
    "financial_status": "受害者的经济/财务状况描述",
    "loss_amount": "被骗金额（数字）",
    "psych_analysis": {{
        "trigger": "被骗的直接触发点（如：急需用钱、情感寂寞、接到恐吓电话）",
        "weakness": "心理弱点（如：贪念、恐惧、轻信、缺乏常识）"
    }},
    "key_behavior": "受害者在被骗过程中的关键操作（如：点击链接、共享屏幕、转账给陌生账户）"
}}
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        res = response.choices[0].message.content
        res = res.replace("```json", "").replace("```", "").strip()
        data = json.loads(res)
        
        age = data["victim_basic"]["age"]
        gender = data["victim_basic"]["gender"]
        occupation = data["victim_basic"]["occupation"]
        financial = data.get("financial_status")
        loss = data.get("loss_amount")
        weakness = data["psych_analysis"]["weakness"]
        behavior = data.get("key_behavior")
        
        # 强制转字符串，避免列表/数组导致报错
        return [
            str(age) if age is not None else None,
            str(gender) if gender is not None else None,
            str(occupation) if occupation is not None else None,
            str(financial) if financial is not None else None,
            str(loss) if loss is not None else None,
            str(weakness) if weakness is not None else None,
            str(behavior) if behavior is not None else None
        ]
    
    except Exception as e:
        return [None] * 7

# ===================== 主程序（修复核心写法） =====================
if __name__ == "__main__":
    print("正在读取 Excel 文件...")
    df = pd.read_excel(FILE_PATH)
    
    cols = ["受害者年龄", "受害者性别", "受害者职业", "受害者经济状况", "被骗金额", "心理弱点","关键问题"]
    for c in cols:
        df[c] = None
    
    print("开始调用 LLM 生成受害者画像...")
    
    for i, row in df.iterrows():
        content = str(row.get("content", ""))
        print(f"处理第 {i+1} 条...")
        
        result = extract_victim_profile(content)

        # ----------------- 这里是修复核心 -----------------
        # 不用 df.loc[i, cols] = result
        # 改用 逐行逐列写入，100%不报错
        df.at[i, cols[0]] = result[0]
        df.at[i, cols[1]] = result[1]
        df.at[i, cols[2]] = result[2]
        df.at[i, cols[3]] = result[3]
        df.at[i, cols[4]] = result[4]
        df.at[i, cols[5]] = result[5]
        df.at[i, cols[6]] = result[6]

        time.sleep(0.5)
    
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"\n✅ 完成！文件已保存：{OUTPUT_FILE}")

正在读取 Excel 文件...
开始调用 LLM 生成受害者画像...
处理第 1 条...
处理第 2 条...
处理第 3 条...
处理第 4 条...
处理第 5 条...
处理第 6 条...
处理第 7 条...
处理第 8 条...
处理第 9 条...
处理第 10 条...
处理第 11 条...
处理第 12 条...
处理第 13 条...
处理第 14 条...
处理第 15 条...
处理第 16 条...
处理第 17 条...
处理第 18 条...
处理第 19 条...
处理第 20 条...
处理第 21 条...
处理第 22 条...
处理第 23 条...
处理第 24 条...
处理第 25 条...
处理第 26 条...
处理第 27 条...
处理第 28 条...
处理第 29 条...
处理第 30 条...
处理第 31 条...
处理第 32 条...
处理第 33 条...
处理第 34 条...
处理第 35 条...
处理第 36 条...
处理第 37 条...
处理第 38 条...
处理第 39 条...
处理第 40 条...
处理第 41 条...
处理第 42 条...
处理第 43 条...
处理第 44 条...
处理第 45 条...
处理第 46 条...
处理第 47 条...
处理第 48 条...
处理第 49 条...
处理第 50 条...
处理第 51 条...
处理第 52 条...
处理第 53 条...
处理第 54 条...
处理第 55 条...
处理第 56 条...
处理第 57 条...
处理第 58 条...
处理第 59 条...
处理第 60 条...
处理第 61 条...
处理第 62 条...
处理第 63 条...
处理第 64 条...
处理第 65 条...
处理第 66 条...
处理第 67 条...
处理第 68 条...
处理第 69 条...
处理第 70 条...
处理第 71 条...
处理第 72 条...
处理第 73 条...
处理第 74 条...
处理第 75 条...
处理第 76 条...
处理第 77 条...
处理第 78 条...
处理第 79 条...
处理第 80 条...
处理第 81 条...
